# Lezione 18 — Similarita' coseno: quanto si somigliano due memorie

**Come si usa questo notebook.** Come sempre. Prerequisito: Lezione 17
(l'incorporatore che trasforma un testo in un vettore fisso). Avere un
vettore per memoria serve a poco senza un modo di **confrontarli**: oggi
la domanda diventa "quanto sono simili due vettori?", con uno strumento
che tornera' in ogni lezione successiva del modulo.

**Cosa saprai fare alla fine:** calcolare la similarita' coseno a mano e
con `sklearn`, spiegare perche' e' invariante alla scala (a differenza
della distanza euclidea), e usarla per trovare, dentro un insieme di
memorie, quelle piu' simili a una memoria data.

## Parte 1 — Teoria: direzione, non lunghezza

Due vettori-frase possono essere confrontati in molti modi. Il piu'
intuitivo, la **distanza euclidea** (Lezione 7: `norm(a - b)`), misura
quanto sono "lontani" i due punti nello spazio — ma e' sensibile alla
**lunghezza** dei vettori: due vettori che puntano nella stessa direzione
ma con norme diverse (uno "piu' forte" dell'altro, per esempio perche' la
frase e' piu' lunga e il mean pooling ha mediato su piu' token) risultano
distanti anche se rappresentano concetti simili.

La **similarita' coseno** risolve questo confrontando solo la
**direzione**:

```
cos(a, b) = (a . b) / (||a|| * ||b||)
```

Il numeratore e' il prodotto scalare (Lezione 7); il denominatore
normalizza per le norme dei due vettori, cancellando l'effetto della
lunghezza. Il risultato e' compreso in `[-1, 1]`: `1` significa stessa
direzione esatta (massimamente simili), `0` significa direzioni
ortogonali (nessuna relazione), `-1` significa direzioni opposte. Per
embedding di testo appresi come nella Lezione 17 i valori tipici stanno
quasi sempre tra `0` e `1`, raramente negativi.

Perche' l'invarianza alla scala conta per gli embedding: due frasi che
esprimono lo stesso concetto con lunghezze diverse (una frase breve e una
piu' elaborata sullo stesso fatto) possono produrre vettori con norme
diverse ma direzione simile — la similarita' coseno li riconosce comunque
vicini, mentre la distanza euclidea penalizzerebbe la differenza di norma
come se fosse una differenza di significato.

In pratica non si scrive la formula a mano: `sklearn.metrics.pairwise.
cosine_similarity` la calcola, tra due insiemi di vettori, in un colpo
solo (una matrice `N x M` di similarita', non solo un numero).

In [1]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

a = np.array([1.0, 2.0, 0.0])
b = np.array([2.0, 4.0, 0.0])   # stessa direzione di a, norma doppia
c = np.array([0.0, 0.0, 3.0])   # direzione ortogonale ad a

# calcolo a mano, dalla formula
coseno_ab = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
coseno_ac = np.dot(a, c) / (np.linalg.norm(a) * np.linalg.norm(c))
print(f'a vs b (stessa direzione, norma diversa): {coseno_ab:.3f}')
print(f'a vs c (direzioni ortogonali)           : {coseno_ac:.3f}')

# stesso risultato con sklearn, su una matrice di vettori
matrice = np.stack([a, b, c])
print('\nmatrice di similarita\' (sklearn):')
print(np.round(cosine_similarity(matrice), 3))

a vs b (stessa direzione, norma diversa): 1.000
a vs c (direzioni ortogonali)           : 0.000

matrice di similarita' (sklearn):
[[1. 1. 0.]
 [1. 1. 0.]
 [0. 0. 1.]]


Leggi l'output: `a` e `b` hanno similarita' coseno `1.0` nonostante `b`
abbia norma doppia di `a` — la scala non conta, solo la direzione. `a` e
`c` hanno similarita' `0.0`: sono ortogonali (nessuna componente in
comune). La distanza euclidea tra `a` e `b`, per confronto, sarebbe
tutt'altro che zero (`b - a = [1, 2, 0]`, norma ~2.24): due misure che
raccontano storie diverse sugli stessi due vettori.

## Parte 2 — Esercizio guidato

Il tuo compito: calcola la similarita' coseno tra `a` e un nuovo vettore
`d = np.array([-1.0, -2.0, 0.0])` (direzione esattamente opposta ad `a`).
Cosa ti aspetti, prima di calcolarlo?

In [2]:
# Scrivi qui: d = np.array([-1.0, -2.0, 0.0]); calcola coseno tra a e d
# (a mano o con cosine_similarity) e stampa il risultato.

pass

### Soluzione spiegata

`d` punta nella direzione esattamente opposta ad `a` (stessi valori, segno
invertito): la similarita' coseno deve essere `-1.0`, il minimo possibile
— il caso limite di "nessuna somiglianza direzionale", anzi il contrario
esatto.

In [3]:
d = np.array([-1.0, -2.0, 0.0])
coseno_ad = np.dot(a, d) / (np.linalg.norm(a) * np.linalg.norm(d))
print(f'a vs d (direzioni opposte): {coseno_ad:.3f}')
assert np.isclose(coseno_ad, -1.0)

a vs d (direzioni opposte): -1.000


## Parte 3 — Il progetto: Memory AI Lab, passo 18 — trovare memorie simili

Ricostruiamo l'incorporatore della Lezione 17 (stesso codice: `Embedding`
addestrato dentro un piccolo classificatore, poi usato fino al pooling) e
lo usiamo per una cosa nuova: dato l'insieme delle memorie di validation,
trovare per ciascuna memoria le sue **piu' simili** per similarita'
coseno. Una verifica di sanita': se l'embedding ha imparato qualcosa di
utile, le memorie dello stesso `type` (episodic/preference/semantic/
unknown) dovrebbero, in media, essere piu' simili tra loro di quanto lo
siano memorie di type diversi — anche se l'embedding non ha mai visto
esplicitamente il type durante l'addestramento del pooling (lo vede solo
indirettamente, tramite la loss di classificazione).

In [4]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import pandas as pd
import keras
from keras.layers import TextVectorization
from pathlib import Path

keras.utils.set_random_seed(42)

processed = Path('..') / 'datasets' / 'processed'
memorie = {n: pd.read_csv(processed / f'memory_{n}.csv') for n in ['train', 'val']}
classi = ['episodic', 'preference', 'semantic', 'unknown']
mappa = {c: i for i, c in enumerate(classi)}
testi = {n: f['text'].astype(str).to_numpy() for n, f in memorie.items()}
target = {n: f['type'].map(mappa).to_numpy() for n, f in memorie.items()}

LUNGHEZZA_SEQUENZA = 24
vettorizzatore = TextVectorization(max_tokens=300, output_mode='int',
                                   output_sequence_length=LUNGHEZZA_SEQUENZA)
vettorizzatore.adapt(testi['train'])
X_seq = {n: vettorizzatore(t).numpy() for n, t in testi.items()}

ingresso = keras.Input(shape=(LUNGHEZZA_SEQUENZA,))
parole = keras.layers.Embedding(input_dim=300, output_dim=16, mask_zero=True)(ingresso)
vettore_frase = keras.layers.GlobalAveragePooling1D(name='sentence_embedding')(parole)
nascosto = keras.layers.Dense(32, activation='relu')(vettore_frase)
nascosto = keras.layers.Dropout(0.3)(nascosto)
uscita = keras.layers.Dense(4, activation='softmax')(nascosto)

classificatore = keras.Model(ingresso, uscita)
classificatore.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
                       metrics=['accuracy'])
classificatore.fit(X_seq['train'], target['train'], epochs=300,
                   validation_data=(X_seq['val'], target['val']),
                   callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                                            restore_best_weights=True)],
                   verbose=0)

incorporatore = keras.Model(classificatore.input,
                            classificatore.get_layer('sentence_embedding').output)
vettori_val = incorporatore(X_seq['val']).numpy()
print('embedding delle memorie di validation:', vettori_val.shape)

2026-07-16 21:18:49.375081: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


embedding delle memorie di validation: (20, 16)


In [5]:
matrice_similarita = cosine_similarity(vettori_val)
np.fill_diagonal(matrice_similarita, -1)  # escludi la memoria stessa dalla ricerca

indice_query = 0
testo_query = memorie['val']['text'].astype(str).to_numpy()[indice_query]
tipo_query = memorie['val']['type'].to_numpy()[indice_query]

k = 3
indici_simili = np.argsort(matrice_similarita[indice_query])[::-1][:k]

print(f'Query [{tipo_query}]: "{testo_query[:70]}"\n')
print(f'Le {k} memorie piu\' simili per similarita\' coseno:')
tipi_val = memorie['val']['type'].to_numpy()
testi_val = memorie['val']['text'].astype(str).to_numpy()
for i in indici_simili:
    sim = matrice_similarita[indice_query, i]
    print(f'  [{sim:.3f}] [{tipi_val[i]}] "{testi_val[i][:70]}"')

Query [semantic]: "Lucia works on la riunione settimanale."

Le 3 memorie piu' simili per similarita' coseno:
  [0.996] [semantic] "Sara works on la riunione settimanale."
  [0.994] [unknown] "Lucia works on il colloquio."
  [0.976] [semantic] "Elena works on il progetto TensorFlow."


In [6]:
# Verifica di sanita': la similarita' media tra memorie dello STESSO type
# e' piu' alta di quella tra memorie di type DIVERSO?
sim_completa = cosine_similarity(vettori_val)  # matrice fresca, diagonale = 1 (una memoria con se' stessa)
stesso_tipo, tipo_diverso = [], []
n = len(tipi_val)
for i in range(n):
    for j in range(i + 1, n):
        (stesso_tipo if tipi_val[i] == tipi_val[j] else tipo_diverso).append(sim_completa[i, j])

print(f'similarita\' media, stesso type : {np.mean(stesso_tipo):.3f}  (n={len(stesso_tipo)})')
print(f'similarita\' media, type diverso: {np.mean(tipo_diverso):.3f}  (n={len(tipo_diverso)})')

similarita' media, stesso type : 0.975  (n=88)
similarita' media, type diverso: 0.058  (n=102)


Guarda i due numeri onestamente: il divario e' netto (memorie dello
stesso type quasi allineate, memorie di type diverso quasi ortogonali) —
coerente con il fatto che l'`Embedding` e' stato addestrato proprio per
classificare il type, quindi la sua geometria riflette quella separazione.
Non leggerlo pero' come "l'embedding capisce il significato": e' un
segnale statistico su un dataset piccolo e su un compito specifico
(classificare 4 categorie), non una garanzia di qualita' semantica
generale, e non vale come prova per ogni singola coppia di memorie. La
prossima lezione visualizza questa stessa struttura in 2D, per vederla
invece di leggerla in due numeri.

## Cosa hai imparato

- La **similarita' coseno** confronta la **direzione** di due vettori,
  non la lunghezza: invariante alla scala, a differenza della distanza
  euclidea.
- Formula: `cos(a, b) = (a . b) / (||a|| * ||b||)`, range `[-1, 1]`
  (embedding di testo appresi: quasi sempre tra `0` e `1` in pratica).
- `sklearn.metrics.pairwise.cosine_similarity` calcola una matrice intera
  di similarita' tra due insiemi di vettori, non solo un numero alla
  volta.
- Una verifica di sanita' onesta (memorie dello stesso type piu' simili in
  media) e' un modo di controllare se un embedding ha imparato qualcosa di
  sensato, senza fidarsi ciecamente del numero di accuratezza del
  classificatore da cui viene.

## Quiz

1. Perche' la similarita' coseno tra `[1, 2, 0]` e `[2, 4, 0]` e' `1.0`
   anche se i due vettori hanno norme diverse?
2. In che caso la similarita' coseno vale `-1`? E `0`?
3. Perche' calcolare la similarita' media tra memorie dello stesso type e
   confrontarla con quella tra type diversi e' un test piu' onesto della
   semplice accuratezza del classificatore, per giudicare la qualita'
   dell'embedding?

<details>
<summary><b>Apri le risposte</b></summary>

1. Perche' i due vettori puntano nella stessa identica direzione (`[2,4,0]
   = 2 * [1,2,0]`): la formula divide il prodotto scalare per il prodotto
   delle norme, cancellando l'effetto della scala. La similarita' coseno
   misura solo l'angolo tra i vettori, non quanto sono "lunghi".
2. `-1` quando i due vettori puntano in direzioni esattamente opposte
   (angolo di 180 gradi); `0` quando sono ortogonali (nessuna componente
   condivisa, angolo di 90 gradi).
3. L'accuratezza del classificatore misura solo se l'ultimo `Dense`
   riesce a separare le classi — puo' essere alta anche con un embedding
   mediocre, se il resto della rete compensa. Confrontare la similarita'
   media intra-type e inter-type guarda direttamente lo spazio degli
   embedding stesso, isolando se e' *quello* ad avere catturato struttura
   utile, indipendentemente da cosa fa il resto della rete.
</details>

## Fonti

- scikit-learn, `cosine_similarity`:
  https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html
- scikit-learn, *Pairwise metrics, Affinities and Kernels*:
  https://scikit-learn.org/stable/modules/metrics.html

## Prossima lezione

Confrontare due memorie alla volta e' utile, ma per **capire** la
struttura di un intero insieme di memorie serve vederla: la prossima
lezione riduce gli embedding a 2 dimensioni (PCA) per visualizzarli.